# Nền tảng 7 — Kỹ thuật huấn luyện quy mô lớn

Bạn đã có vòng lặp huấn luyện và các optimizer (SGD, momentum, RMSprop, AdamW). Notebook này bổ sung những thứ
nằm giữa "vòng lặp trên giấy" và lệnh mà project thật sự chạy trên Kaggle:

```bash
NANOCHAT_DTYPE=float16 torchrun --nproc_per_node=1 -m scripts.base_train \
    --depth=8 --max-seq-len=840 --device-batch-size=32 --total-batch-size=53760 \
    --scaling-batch-size=65536 --num-iterations=7629 --window-pattern=L
```

Bốn cơ chế đứng sau các tham số đó:

1. **Lịch learning rate** — vì sao không để learning rate cố định.
2. **Tích luỹ gradient** — `device-batch-size` khác `total-batch-size` nghĩa là gì và vì sao nó *đúng về mặt toán*.
3. **Độ chính xác hỗn hợp** — `NANOCHAT_DTYPE=float16`, GradScaler, và vì sao T4 buộc phải dùng fp16 chứ không
   phải bf16.
4. **Huấn luyện phân tán** — DDP hoạt động ra sao, tốn bao nhiêu băng thông, và vì sao project **không** dùng nó
   trên 2×T4.

Mỗi cơ chế đều có một cell kiểm chứng bằng số trên CPU.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
RUNS = ROOT / "kaggle" / "outputs"
torch.manual_seed(0)


def bieu_do(ys, cao=9, rong=72, nhan=""):
    """Vẽ một đường cong bằng ký tự, đủ để thấy hình dạng mà không cần matplotlib."""
    b = [ys[round(i * (len(ys) - 1) / (rong - 1))] for i in range(rong)]
    lo, hi = min(b), max(b)
    khung = [[" "] * rong for _ in range(cao)]
    for x, y in enumerate(b):
        muc = 0 if hi == lo else round((y - lo) / (hi - lo) * (cao - 1))
        khung[cao - 1 - muc][x] = "•"
    print(f"  {hi:8.4f} ┤" + "".join(khung[0]))
    for hang in khung[1:-1]:
        print(f"  {'':8s} │" + "".join(hang))
    print(f"  {lo:8.4f} ┤" + "".join(khung[-1]) + f"   {nhan}")


print("torch", torch.__version__)

## 1. Lịch learning rate

Learning rate cố định gặp hai vấn đề ở hai đầu quá trình huấn luyện:

- **Đầu**: trọng số ngẫu nhiên, gradient rất lớn và hướng của chúng gần như vô nghĩa. Với Adam còn thêm một lý do
  riêng: ước lượng moment bậc hai $v_t$ tích luỹ từ rất ít mẫu nên không đáng tin, làm bước đi
  $\eta\,\hat m_t/(\sqrt{\hat v_t}+\epsilon)$ có thể rất lớn. Cách xử lý là **warmup**: tăng learning rate tuyến
  tính từ 0 trong vài chục bước.
- **Cuối**: muốn hội tụ vào đáy thì bước phải nhỏ dần, nếu không model cứ nảy quanh cực tiểu. Cách xử lý là
  **warmdown** (hay decay).

nanochat dùng lịch ba đoạn — tăng tuyến tính, giữ nguyên, giảm tuyến tính — thay vì cosine:

$$\text{mult}(t) = \begin{cases}
(t+1)/T_{\text{warmup}} & t < T_{\text{warmup}} \\
1 & T_{\text{warmup}} \le t \le T - T_{\text{warmdown}} \\
\rho \cdot 1 + (1-\rho)\,f_{\text{cuối}}, \quad \rho = \frac{T - t}{T_{\text{warmdown}}} & \text{còn lại}
\end{cases}$$

**Ký hiệu mới:** $\text{mult}(t)$ — hệ số nhân vào learning rate ở bước $t$; $\rho$ — phần còn lại của đoạn
warmdown, đi từ 1 về 0 (không phải hệ số tương quan).

Tham số của project: $T = 7629$ bước, $T_{\text{warmup}} = 40$, warmdown chiếm 65% tổng số bước,
$f_{\text{cuối}} = 0{,}05$. Dạng này ("warmup–stable–decay") có ưu điểm thực dụng so với cosine: đoạn giữa phẳng
nên **cắt ngắn hoặc kéo dài** lịch đều được mà không phải tính lại toàn bộ đường cong.

In [ ]:
T, WARMUP, WARMDOWN_RATIO, FINAL_FRAC = 7629, 40, 0.65, 0.05


def lr_mult(it, T=T, warmup=WARMUP, warmdown_ratio=WARMDOWN_RATIO, final_frac=FINAL_FRAC):
    warmdown = round(warmdown_ratio * T)
    if it < warmup:
        return (it + 1) / warmup
    if it <= T - warmdown:
        return 1.0
    tien = (T - it) / warmdown
    return tien * 1.0 + (1 - tien) * final_frac


lich = [lr_mult(t) for t in range(T)]
bieu_do(lich, nhan=f"bước 0 -> {T}")
print(f"\n  bước    0: {lr_mult(0):.4f}   (warmup bắt đầu từ 1/40)")
print(f"  bước   39: {lr_mult(39):.4f}   (kết thúc warmup)")
print(f"  bước 2670: {lr_mult(2670):.4f}   (điểm bắt đầu warmdown = T - 0,65T)")
print(f"  bước {T - 1}: {lr_mult(T - 1):.4f}   (còn {FINAL_FRAC:.0%} learning rate gốc)")

print("\nlearning rate tuyệt đối của bốn nhóm tham số (nhân với mult ở trên):")
cfg = json.loads((RUNS / "results-v6" / "results" / "bpe-nfc_d8_s0.json").read_text())["user_config"]
for k in ("embedding_lr", "unembedding_lr", "matrix_lr", "scalar_lr"):
    print(f"  {k:16s} = {cfg[k]}")

Muon còn có **lịch momentum** riêng: tăng từ 0,85 lên 0,97 trong 400 bước đầu, giữ 0,97, rồi giảm về 0,90 trong
đoạn warmdown. Momentum cao giúp trung bình hoá nhiễu gradient ở giữa quá trình; hạ xuống ở cuối để model phản
ứng nhanh hơn với gradient hiện tại khi đang tinh chỉnh.

In [ ]:
def muon_momentum(it, T=T, warmdown_ratio=WARMDOWN_RATIO):
    warmdown = round(warmdown_ratio * T)
    bat_dau = T - warmdown
    if it < 400:
        f = it / 400
        return (1 - f) * 0.85 + f * 0.97
    if it >= bat_dau:
        tien = (it - bat_dau) / warmdown
        return 0.97 * (1 - tien) + 0.90 * tien
    return 0.97


bieu_do([muon_momentum(t) for t in range(T)], cao=7, nhan="momentum của Muon")

print("\nweight decay của Muon giảm theo cosine về 0:")
bieu_do([0.5 * (1 + math.cos(math.pi * t / T)) for t in range(T)], cao=7, nhan="hệ số weight decay")

## 2. Kích thước batch và quy tắc chia tỷ lệ learning rate

Gradient tính trên một batch là **ước lượng** của gradient thật trên toàn bộ dữ liệu. Theo đúng công thức ở
notebook 02 mục 3,

$$\operatorname{Var}(\hat g_B) = \frac{\sigma_g^2}{B}$$

**Ký hiệu mới:** $\hat g_B$ — gradient trung bình trên một batch $B$ mẫu; $\sigma_g^2$ — phương sai gradient của
một mẫu. Đây đúng là $\operatorname{Var}(\bar X) = \sigma^2/n$ ở notebook 02.

nên batch lớn gấp 4 thì độ lệch chuẩn của nhiễu gradient giảm 2 lần. Gradient ít nhiễu hơn thì chịu được bước đi
dài hơn, và đó là lý do learning rate nên tăng theo batch:

| Optimizer | Quy tắc | Lý do vắn tắt |
|---|---|---|
| SGD | $\eta \propto B$ (tuyến tính) | tổng độ dịch chuyển mỗi epoch giữ nguyên |
| Adam/AdamW | $\eta \propto \sqrt{B}$ | giữ nguyên "độ nhiễu" của quỹ đạo tối ưu khi batch đổi (phân tích qua phương trình vi phân ngẫu nhiên, Malladi et al. 2022) |

Cả hai là **quy tắc kinh nghiệm**, chỉ đúng khi batch còn nhỏ hơn một ngưỡng tới hạn; vượt ngưỡng đó thì tăng
batch không cho tăng learning rate tương ứng nữa (McCandlish et al. 2018, xem nguồn cuối notebook). nanochat
dùng quy tắc căn với mốc $B_{\text{ref}} = 2^{19}$ token, và project **ghim** mốc đó qua
`--scaling-batch-size` để tokenizer nén khác nhau không kéo theo learning rate khác nhau (notebook 03 mục 8).

Cell dưới đo trực tiếp $\operatorname{Var}(\hat g_B) \propto 1/B$ trên một model nhỏ.

In [ ]:
d, N = 64, 4096
W = torch.randn(d, 1, requires_grad=True)
X = torch.randn(N, d)
y = X @ torch.randn(d, 1) + 0.5 * torch.randn(N, 1)


def grad_batch(idx):
    W.grad = None
    ((X[idx] @ W - y[idx]) ** 2).mean().backward()
    return W.grad.clone()


print("  B    | sd(gradient) trên 200 batch | sd × √B")
for B in (8, 32, 128, 512):
    gs = torch.stack([grad_batch(torch.randint(0, N, (B,))) for _ in range(200)])
    sd = gs.std(0).mean().item()
    print(f" {B:4d}  | {sd:27.5f} | {sd * math.sqrt(B):7.4f}")
print("\ncột cuối gần như không đổi => sd ∝ 1/√B, đúng như công thức.")

B_REF = 2 ** 19
print(f"\nquy tắc căn của nanochat, mốc B_ref = {B_REF:,} token:")
for batch, ten in ((64 * 1024, "bpe-nfc: 64 × 1024"), (64 * 840, "super-nfc: 64 × 840")):
    print(f"  {ten:22s} = {batch:6,} token -> η nhân {math.sqrt(batch / B_REF):.4f}")

## 3. Tích luỹ gradient

Project muốn mỗi bước cập nhật dùng 64 chuỗi, nhưng forward/backward cả 64 chuỗi cùng lúc thì bộ nhớ activation
vượt 16GB của T4 (32 chuỗi đã tốn 7,6–9,0 GB ở d8, xem mục 6).
Giải pháp: chia thành các **micro-batch**, chạy forward/backward từng cái, cộng dồn gradient, rồi mới cập nhật.

Điều này **đúng chính xác**, không phải xấp xỉ, vì gradient là toán tử tuyến tính theo tổng loss:

$$\nabla_\theta \frac{1}{B}\sum_{i=1}^{B} \ell_i = \frac{1}{K}\sum_{k=1}^{K} \nabla_\theta \left(\frac{1}{B/K}\sum_{i \in \text{micro}_k} \ell_i\right)$$

**Ký hiệu mới**
- $\nabla_\theta$ — gradient theo toàn bộ tham số $\theta$
- $\ell_i$ — loss của mẫu $i$
- $K$ — số micro-batch; $\text{micro}_k$ — tập mẫu của micro-batch thứ $k$

Đẳng thức trên cần một điều kiện: các micro-batch có **cùng số token được tính loss**. Khi pretrain với chuỗi
cố định độ dài và không có token bị bỏ qua thì điều kiện đó đúng; khi có đệm hoặc mặt nạ loss (như SFT ở
notebook 09), trung bình của các trung bình không còn bằng trung bình chung, và phép tích luỹ chỉ còn xấp xỉ.

Chi tiết dễ sai nằm ở hệ số: PyTorch **cộng dồn** gradient qua các lần `.backward()`, nên phải chia loss cho số
micro-batch, đúng như dòng trong `base_train.py`:

```python
loss = loss / grad_accum_steps  # each .backward() is a grad sum => normalize loss here
```

Cell dưới kiểm chứng bằng cách so gradient tích luỹ với gradient tính một lần trên cả batch.

In [ ]:
mo_hinh = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
x, y = torch.randn(64, 16), torch.randn(64, 1)

mo_hinh.zero_grad()
F.mse_loss(mo_hinh(x), y).backward()
g_mot_lan = torch.cat([p.grad.flatten().clone() for p in mo_hinh.parameters()])

K = 4
mo_hinh.zero_grad()
for k in range(K):
    lat = slice(k * 16, (k + 1) * 16)
    (F.mse_loss(mo_hinh(x[lat]), y[lat]) / K).backward()        # chia cho K rồi cộng dồn
g_tich_luy = torch.cat([p.grad.flatten().clone() for p in mo_hinh.parameters()])

print(f"sai khác lớn nhất: {(g_mot_lan - g_tich_luy).abs().max():.3e}  -> bằng nhau đến sai số dấu phẩy động")

mo_hinh.zero_grad()
for k in range(K):
    lat = slice(k * 16, (k + 1) * 16)
    F.mse_loss(mo_hinh(x[lat]), y[lat]).backward()               # QUÊN chia cho K
g_quen = torch.cat([p.grad.flatten().clone() for p in mo_hinh.parameters()])
print(f"nếu quên chia cho K: gradient lớn gấp {(g_quen.norm() / g_mot_lan.norm()):.1f} lần"
      f" -> tương đương learning rate gấp {K} lần")

print("\ncấu hình thật của project ở d8 (super-nfc):")
tong_batch, thiet_bi_batch, seq = 53760, 32, 840
print(f"  total-batch-size   = {tong_batch:,} token = 64 chuỗi × {seq}")
print(f"  device-batch-size  = {thiet_bi_batch} chuỗi = {thiet_bi_batch * seq:,} token mỗi lần forward")
print(f"  => số micro-batch  = {tong_batch // (thiet_bi_batch * seq)}")

## 4. Độ chính xác hỗn hợp

Ba kiểu số thực thường gặp, cùng 1 bit dấu nhưng chia phần mũ/phần định trị khác nhau:

| Kiểu | Bit mũ | Bit định trị | Khoảng giá trị | Độ phân giải tương đối |
|---|---|---|---|---|
| fp32 | 8 | 23 | $\sim10^{\pm38}$ | $\sim10^{-7}$ |
| fp16 | 5 | 10 | $6\times10^{-5}$ đến $65\,504$ | $\sim10^{-3}$ |
| bf16 | 8 | 7 | như fp32 | $\sim10^{-2}$ |

Điểm mấu chốt: **bf16 có cùng khoảng giá trị với fp32**, chỉ kém độ phân giải; **fp16 thì ngược lại** — phân giải
tốt hơn bf16 nhưng khoảng giá trị hẹp, nên gradient nhỏ dễ bị **tràn dưới** (underflow) thành 0 và gradient lớn
dễ **tràn trên** (overflow) thành `inf`.

Vì sao điều này quan trọng với project: T4 là kiến trúc Turing, **không có** bf16 trên phần cứng. nanochat tự dò
sẽ thấy "không bf16" và rơi về fp32 — khi đó không dùng được tensor core: đỉnh tính toán của T4 ở fp32 chỉ là
8,1 TFLOPS so với 65 TFLOPS ở fp16 trên tensor core, và activation tốn gấp đôi bộ nhớ. Project ép
`NANOCHAT_DTYPE=float16` để dùng tensor core, và phải kèm GradScaler.

In [ ]:
print("giới hạn của fp16:")
info = torch.finfo(torch.float16)
print(f"  lớn nhất {info.max} | nhỏ nhất (chuẩn hoá) {info.tiny:.2e} | eps {info.eps}")
print(f"  bf16: lớn nhất {torch.finfo(torch.bfloat16).max:.3e} | eps {torch.finfo(torch.bfloat16).eps}")

print("\ntràn trên và tràn dưới:")
for v in (1e-8, 1e-5, 1.0, 1e4, 1e5):
    x16 = torch.tensor(v, dtype=torch.float16)
    xbf = torch.tensor(v, dtype=torch.bfloat16)
    print(f"  {v:8.0e} -> fp16 {x16.item():12.3e} {'(mất thành 0)' if x16 == 0 else '(inf!)' if x16.isinf() else ''}"
          f" | bf16 {xbf.item():.3e}")

print("\nđộ phân giải: cộng một lượng nhỏ vào 1.0")
for v in (1e-2, 1e-3, 1e-4):
    a16 = (torch.tensor(1.0, dtype=torch.float16) + torch.tensor(v, dtype=torch.float16)).item()
    abf = (torch.tensor(1.0, dtype=torch.bfloat16) + torch.tensor(v, dtype=torch.bfloat16)).item()
    print(f"  1 + {v:.0e}: fp16 -> {a16:.6f} | bf16 -> {abf:.6f}  {'(bf16 mất luôn)' if abf == 1.0 else ''}")

### Loss scaling và GradScaler

Cách chữa tràn dưới rất gọn: nhân loss với một hằng số $S$ trước khi backward. Vì gradient tuyến tính theo loss,

$$\nabla_\theta (S \cdot \ell) = S \cdot \nabla_\theta \ell$$

nên mọi gradient được nâng lên $S$ lần, thoát khỏi vùng tràn dưới. Trước khi optimizer cập nhật, chia gradient
lại cho $S$ (bước `unscale_`).

$S$ phải đủ lớn để cứu gradient nhỏ nhưng không quá lớn tới mức gradient lớn tràn trên. Không có giá trị đúng cố
định, nên `torch.amp.GradScaler` dò **động**:

1. Bắt đầu với $S = 65\,536$.
2. Sau backward, kiểm tra gradient có `inf`/`nan` không.
3. Nếu có: **bỏ qua** bước cập nhật này và chia $S$ cho 2.
4. Nếu không có trong 2.000 bước liên tiếp: nhân $S$ với 2 để thử vùng an toàn hơn.

Bước 3 là điểm đáng nhớ: một vài bước đầu thường bị bỏ qua — chuyện bình thường, không phải lỗi.

Cell dưới cài lại đúng vòng dò đó bằng tay.

In [ ]:
class ScalerDoChoi:
    def __init__(self, S=65536.0, chu_ky_tang=2000):
        self.S, self.chu_ky_tang, self.sach = S, chu_ky_tang, 0

    def scale(self, loss):
        return loss * self.S

    def buoc(self, grads):
        """Trả về (có_cập_nhật, gradient đã chia lại)."""
        if any(not torch.isfinite(g).all() for g in grads):
            self.S /= 2
            self.sach = 0
            return False, None
        self.sach += 1
        if self.sach >= self.chu_ky_tang:
            self.S *= 2
            self.sach = 0
        return True, [g / self.S for g in grads]


scaler = ScalerDoChoi()
print("mô phỏng: gradient thật rất nhỏ (1e-7), thỉnh thoảng có một bước gradient lớn")
print("  bước | S        | gradient sau scale | cập nhật?")
for buoc in range(8):
    g_that = torch.tensor([1e-7], dtype=torch.float32)
    if buoc in (2, 5):
        g_that = torch.tensor([3.0], dtype=torch.float32)        # bước gradient lớn
    g16 = (g_that * scaler.S).half()                             # gradient tính ở fp16
    ok, g_chia = scaler.buoc([g16])
    print(f"  {buoc:4d} | {scaler.S:8.0f} | {g16.item():18.4g} | {'có' if ok else 'BỎ QUA (tràn)'}")

print("\nkhông có loss scaling, gradient nhỏ ở fp16:")
for g in (1e-7, 1e-8, 1e-9):
    x = torch.tensor([g], dtype=torch.float16).item()
    print(f"  {g:.0e} -> {x:.3e}  {'MẤT TRẮNG (= 0)' if x == 0 else '(số dưới chuẩn, sai số tương đối lớn)'}")
print("  ngưỡng: fp16 về 0 khi giá trị dưới ~6e-8; vùng 6e-8 đến 6e-5 là số dưới chuẩn, độ chính xác kém dần.")

### Trọng số chủ ở fp32

Chi tiết cuối của độ chính xác hỗn hợp: **trọng số vẫn giữ ở fp32**, chỉ phép nhân ma trận chạy ở fp16/bf16.
Trong nanochat điều này nằm trong lớp `Linear` tự viết:

```python
class Linear(nn.Linear):
    # nn.Linear that casts weights to match input dtype in forward.
    # Master weights stay fp32 for optimizer precision, but matmuls run in the activation dtype.
```

Lý do: một bước cập nhật điển hình có độ lớn $\eta \cdot g \sim 10^{-6}$ so với trọng số $\sim 10^{-1}$. Tỷ lệ
$10^{-5}$ nhỏ hơn eps của fp16 ($10^{-3}$), nên nếu trọng số ở fp16 thì phép cộng **không thay đổi gì cả** và
model đứng yên.

In [ ]:
w32 = torch.tensor([0.1], dtype=torch.float32)
w16 = torch.tensor([0.1], dtype=torch.float16)
print(f"giá trị ban đầu: fp32 lưu 0.1 thành {w32.item():.8f} | fp16 lưu thành {w16.item():.8f}")
print(f"khoảng cách giữa hai số fp16 liền nhau quanh 0,1: {torch.finfo(torch.float16).eps * 0.0625:.2e}\n")
buoc = 1e-6
for _ in range(1000):
    w32 = w32 - buoc
    w16 = w16 - torch.tensor([buoc], dtype=torch.float16)
print(f"sau 1.000 bước cập nhật cỡ {buoc}:")
print(f"  fp32: 0.1 -> {w32.item():.6f}   (đúng như mong đợi: giảm 0,001)")
print(f"  fp16: {torch.tensor([0.1], dtype=torch.float16).item():.6f} -> {w16.item():.6f}"
      f"   {'<- ĐỨNG YÊN: mỗi bước 1e-6 nhỏ hơn nửa khoảng cách 6e-5, bị làm tròn về chỗ cũ' if w16.item() == torch.tensor([0.1], dtype=torch.float16).item() else ''}")
print("\n=> trọng số chủ phải ở fp32; đó là lý do 'mixed' precision chứ không phải 'half' precision.")

## 5. Vì sao hai lần train cùng seed vẫn ra số khác nhau

Notebook 02 mục 11 dùng chênh lệch giữa hai seed làm thanh nhiễu. Nhưng ngay cả **cùng seed** trên cùng máy, kết
quả cũng không nhất thiết trùng khít, và lý do nằm ở số học:

- Phép cộng dấu phẩy động **không kết hợp**: $(a + b) + c \ne a + (b + c)$ khi các số chênh lệch độ lớn.
- Kernel GPU chia việc theo thread và cộng dồn theo thứ tự phụ thuộc lịch chạy; thứ tự đó không cố định.
- fp16 có eps $10^{-3}$, nên sai số tích luỹ nhanh hơn hẳn fp32.

Cell dưới minh hoạ điểm thứ nhất — nguồn gốc của mọi thứ còn lại.

In [ ]:
a = torch.tensor([2048.0], dtype=torch.float16)
b = torch.tensor([1.0], dtype=torch.float16)
print("ở lân cận 2048, khoảng cách giữa hai số fp16 liền nhau là 2, nên cộng 1 bị làm tròn:")
print(f"  (2048 + 1) + 1 = {((a + b) + b).item():.1f}")
print(f"  2048 + (1 + 1) = {(a + (b + b)).item():.1f}   <- khác nhau, chỉ vì thứ tự cộng")

x = torch.randn(20000, dtype=torch.float16)
tuan_tu = torch.tensor([0.0], dtype=torch.float16)
for v in x:                                              # cộng tuần tự, như một thread duy nhất
    tuan_tu = tuan_tu + v
theo_khoi = sum((x[i:i + 500].sum() for i in range(0, len(x), 500)),
                torch.tensor(0.0, dtype=torch.float16))  # cộng theo khối, như nhiều thread
print(f"\ncộng 20.000 số fp16:")
print(f"  tuần tự            : {tuan_tu.item():9.4f}")
print(f"  theo khối 500      : {theo_khoi.item():9.4f}")
print(f"  đúng (fp64)        : {x.double().sum().item():9.4f}")
print("\n=> cách chia việc quyết định kết quả. Trên GPU cách chia đó phụ thuộc lịch chạy của thread,")
print("   nên hai lần train cùng seed vẫn lệch nhau ở mức này, rồi sai lệch tích luỹ qua hàng nghìn bước.")

## 6. DDP: huấn luyện trên nhiều GPU

**Distributed Data Parallel**: mỗi GPU (gọi là *rank*) giữ **một bản sao đầy đủ** của model, nhận một phần khác
nhau của batch, tính gradient riêng, rồi tất cả các rank **trung bình gradient với nhau** trước khi cập nhật. Vì
gradient được trung bình, kết quả tương đương với việc chạy một batch lớn gấp `world_size` lần trên một GPU.

Phép trung bình đó là **all-reduce**: sau lệnh, mọi rank đều có cùng một kết quả. Cài đặt phổ biến là *ring
all-reduce*, chia tensor thành `world_size` mảnh và chuyền vòng tròn, tổng lượng dữ liệu mỗi rank phải gửi là

$$\text{byte gửi} = 2 \cdot \frac{W - 1}{W} \cdot N \cdot \text{(byte mỗi tham số)}$$

**Vì sao $\frac{W-1}{W}$:** mỗi rank đã giữ sẵn $\frac{1}{W}$ phần của mình, chỉ phải nhận phần còn lại.

với $N$ là số tham số và $W$ số rank (hệ số 2 vì có hai vòng: reduce-scatter rồi all-gather).

Cell đầu mô phỏng DDP bằng cách chia batch làm hai "rank" trên CPU và kiểm tra gradient trung bình đúng bằng
gradient của cả batch.

In [ ]:
mo_hinh = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
x, y = torch.randn(64, 16), torch.randn(64, 1)

mo_hinh.zero_grad()
F.mse_loss(mo_hinh(x), y).backward()
g_mot_gpu = [p.grad.clone() for p in mo_hinh.parameters()]

grads_rank = []
for r in range(2):                                     # hai "rank", mỗi rank nửa batch
    lat = slice(r * 32, (r + 1) * 32)
    mo_hinh.zero_grad()
    F.mse_loss(mo_hinh(x[lat]), y[lat]).backward()
    grads_rank.append([p.grad.clone() for p in mo_hinh.parameters()])

g_ddp = [(a + b) / 2 for a, b in zip(*grads_rank)]      # all-reduce trung bình
print("sai khác giữa DDP 2 rank và một GPU chạy cả batch:")
print(f"  {max((a - b).abs().max().item() for a, b in zip(g_mot_gpu, g_ddp)):.3e}")
print("  => DDP không phải xấp xỉ: nó là đúng cùng một phép tính, chia ra cho nhiều máy.")

In [ ]:
print("chi phí truyền thông của DDP trên 2 GPU, gradient fp32:\n")
print(" model | tham số      | byte all-reduce mỗi bước | thời gian trên PCIe 3.0 x16 (~12 GB/s)")
for depth, N in ((6, 41_459_858), (8, 74_514_666), (10, 121_119_066)):
    byte = 2 * (2 - 1) / 2 * N * 4
    print(f" d{depth:<4} | {N:12,} | {byte / 1e6:22.1f} MB | {byte / 12e9 * 1000:20.0f} ms")

tp = json.loads((RUNS / "results-v9" / "results" / "throughput_d10.json").read_text())
dt_ms = tp["bpe-nfc_d10_s0"]["median_dt_ms"]
byte_d10 = 2 * 0.5 * 121_119_066 * 4
print(f"\nthời gian một bước đo được ở d10 (1 GPU): {dt_ms:.0f} ms")
print(f"nếu chia đôi công việc cho 2 GPU: tính toán còn ~{dt_ms / 2:.0f} ms, cộng ~{byte_d10 / 12e9 * 1000:.0f} ms"
      f" truyền thông = {dt_ms / 2 + byte_d10 / 12e9 * 1000:.0f} ms")
print(f"tăng tốc lý thuyết: {dt_ms / (dt_ms / 2 + byte_d10 / 12e9 * 1000):.2f}×")

Con số trên là **cận trên lạc quan**: nó giả định truyền thông chồng hoàn toàn lên tính toán được bỏ qua, không
tính chi phí khởi động NCCL, và giả định băng thông PCIe đạt đỉnh (Kaggle không nối NVLink giữa hai T4).

Nhưng lập luận quyết định cho project không phải tốc độ của một lượt mà là **tổng thời gian cho cả hàng đợi**:
có nhiều lượt train độc lập. Gọi $s$ là tăng tốc DDP của một lượt trên 2 GPU ($s < 2$ vì luôn có truyền thông).

- **Mỗi GPU một lượt, song song**: 2 lượt xong sau thời gian $t$ — thông lượng 2 lượt mỗi $t$.
- **DDP, lần lượt**: mỗi lượt mất $t/s$, 2 lượt mất $2t/s > t$.

Nên khi số lượt chạy còn chẵn và có độ dài tương đương, chạy song song **luôn** thắng DDP, với bất kỳ $s < 2$.
DDP chỉ có lợi ở phần **đuôi**: khi một GPU đã xong việc còn GPU kia vẫn chạy một lượt dài, GPU rảnh bị lãng
phí — lúc đó chuyển lượt còn lại sang DDP có thể rút ngắn tổng thời gian. Đây đúng là tình huống thời gian chết
khi hai hàng đợi có tổng độ dài lệch nhau, và cách xử lý rẻ hơn là **cân hàng đợi** (mỗi GPU một lượt BPE và một
lượt SuperBPE) thay vì chuyển sang DDP.

Ngoại lệ duy nhất: khi một điều kiện **không vừa bộ nhớ** một GPU. Không phải trường hợp của project, vì đỉnh bộ
nhớ đo được chỉ 6–9 GB trên card 16GB.

In [ ]:
print("đỉnh bộ nhớ đo được (peak_mib) của mọi lượt train:\n")
print(" run                 | peak MiB | % của T4 16GB")
for f in sorted(RUNS.glob("results-v*/results/throughput_d*.json")):
    for ten, v in json.loads(f.read_text()).items():
        print(f" {ten:19s} | {v['peak_mib']:8,.0f} | {v['peak_mib'] / 15360:13.0%}")
print("\nCòn dư chỗ, nên lý do duy nhất để dùng DDP (không vừa bộ nhớ) không tồn tại ở đây.")

Một quan sát phụ từ bảng trên: các lượt SuperBPE tốn **ít bộ nhớ hơn** BPE (7,6 GB so với 9,0 GB ở d8). Nguyên
nhân là `max_seq_len` ngắn hơn (840 so với 1024): bộ nhớ activation tỷ lệ thuận với số token trong micro-batch.
(Ma trận điểm attention $T \times T$ chỉ tốn bộ nhớ $\propto T^2$ nếu kernel thật sự dựng nó ra; các kernel
hiệu quả bộ nhớ của SDPA tính theo khối nên không giữ toàn bộ ma trận.) Còn d10 tốn **ít** hơn d8 vì dùng
`device-batch-size = 16` thay vì 32 (`DEVICE_BATCH` trong `vitok/conditions.py`), bù lại bằng 4 micro-batch.

$$\text{bộ nhớ activation} \approx B \cdot T \cdot d \cdot L \cdot (\text{số tensor trung gian}) + \text{phần attention}$$

**Ký hiệu mới:** $B$ — số chuỗi trong micro-batch; số tensor trung gian — số kết quả mà backward phải giữ lại mỗi
lớp.

## 7. Ghép lại: một bước train đầy đủ

Vòng lặp thật của `base_train.py`, rút gọn còn phần cốt lõi:

```python
for step in range(num_iterations):
    for micro_step in range(grad_accum_steps):          # tích luỹ gradient
        loss = model(x, y)
        loss = loss / grad_accum_steps
        scaler.scale(loss).backward()                   # loss scaling cho fp16
    scaler.unscale_(optimizer)                          # chia gradient lại cho S
    for group in optimizer.param_groups:                # lịch learning rate
        group["lr"] = group["base_lr"] * get_lr_multiplier(step)
    scaler.step(optimizer)                              # bỏ qua nếu gradient có inf/nan
    scaler.update()                                     # điều chỉnh S
    optimizer.zero_grad(set_to_none=True)
```

Cell dưới chạy đúng cấu trúc đó trên một model nhỏ ở fp32 để bạn thấy mọi mảnh ghép hoạt động cùng nhau.

In [ ]:
torch.manual_seed(0)
mo_hinh = torch.nn.Sequential(torch.nn.Linear(16, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1))
opt = torch.optim.AdamW(mo_hinh.parameters(), lr=1e-2)
for g in opt.param_groups:
    g["base_lr"] = g["lr"]

X, Y = torch.randn(512, 16), torch.randn(512, 1)
T_NHO, K = 300, 4
lich_loss = []
for buoc in range(T_NHO):
    for k in range(K):
        idx = torch.randint(0, len(X), (32,))
        (F.mse_loss(mo_hinh(X[idx]), Y[idx]) / K).backward()
    for g in opt.param_groups:
        g["lr"] = g["base_lr"] * lr_mult(buoc, T=T_NHO, warmup=20)
    opt.step()
    opt.zero_grad(set_to_none=True)
    with torch.no_grad():
        lich_loss.append(F.mse_loss(mo_hinh(X), Y).item())

print(f"loss đầu {lich_loss[0]:.4f} -> cuối {lich_loss[-1]:.4f}")
bieu_do(lich_loss, nhan="loss trên toàn bộ dữ liệu")
bieu_do([lr_mult(t, T=T_NHO, warmup=20) for t in range(T_NHO)], cao=6, nhan="hệ số learning rate")

## 8. Tóm tắt

| Cơ chế | Công thức / quy tắc | Con số của project |
|---|---|---|
| Lịch LR | warmup tuyến tính → phẳng → warmdown tuyến tính | 40 bước warmup, warmdown 65%, còn 5% |
| Momentum Muon | 0,85 → 0,97 → 0,90 | chuyển ở bước 400 |
| Nhiễu gradient | $\operatorname{Var}(\hat g_B) = \sigma_g^2/B$ | cơ sở của quy tắc $\eta \propto \sqrt{B}$ |
| Tích luỹ gradient | chia loss cho $K$ rồi cộng dồn | $K = 2$ ở d8 |
| fp16 | khoảng $[6\times10^{-5}, 65504]$, eps $10^{-3}$ | bắt buộc trên T4 (không có bf16) |
| Loss scaling | $\nabla(S\ell) = S\nabla\ell$, $S$ dò động | GradScaler, bắt đầu 65.536 |
| Trọng số chủ | giữ fp32, matmul ở fp16 | lớp `Linear` của nanochat |
| DDP all-reduce | $2\frac{W-1}{W}N \times$ (byte/tham số) mỗi bước | ~484 MB ở d10 với 2 GPU, ~40 ms trên PCIe |

## 9. Câu hỏi tự kiểm

1. Vì sao Adam cần warmup nhiều hơn SGD?
2. Batch tăng từ 16 lên 64, learning rate nên nhân bao nhiêu với AdamW? Với SGD?
3. Nếu quên chia loss cho số micro-batch, điều gì xảy ra, và nó tương đương với sai lầm nào khác?
4. Vì sao bf16 không cần GradScaler còn fp16 thì cần?
5. Vì sao trọng số phải giữ ở fp32 dù matmul chạy fp16?
6. GradScaler bỏ qua vài bước đầu. Đó là lỗi hay là hành vi đúng?
7. Tính lượng byte all-reduce mỗi bước cho d8 trên 4 GPU, gradient fp32.
8. Project có 4 điều kiện và 2 GPU. Giải thích vì sao "mỗi GPU một điều kiện" thắng DDP, và trong trường hợp nào
   kết luận đó đảo ngược.

**Đáp án gợi ý**

1. Vì bước đi của Adam được chia cho $\sqrt{\hat v_t}$, mà $\hat v_t$ ở vài bước đầu ước lượng từ rất ít mẫu nên
   không đáng tin, dễ cho bước rất lớn.
2. AdamW: $\sqrt{64/16} = 2$ lần. SGD: 4 lần.
3. Gradient lớn gấp $K$ lần, tương đương nhân learning rate với $K$ — thường làm loss phân kỳ.
4. Vì bf16 có cùng khoảng giá trị với fp32 nên gradient nhỏ không bị tràn dưới; fp16 có khoảng hẹp nên cần nâng
   gradient lên trước khi backward.
5. Vì bước cập nhật $\eta g \sim 10^{-6}$ so với trọng số $\sim 10^{-1}$ cho tỷ lệ $10^{-5}$, nhỏ hơn eps của
   fp16 ($10^{-3}$), nên phép cộng không thay đổi gì.
6. Hành vi đúng: nó đang dò giá trị $S$ an toàn, mỗi lần tràn thì giảm một nửa.
7. $2 \cdot \frac{3}{4} \cdot 74{,}5\text{M} \cdot 4 \approx 447$ MB.
8. Vì chạy song song hai điều kiện cho tăng tốc đúng $2\times$ mà không tốn truyền thông, trong khi DDP luôn mất
   một phần cho all-reduce. Đảo ngược khi một điều kiện không vừa bộ nhớ một GPU, hoặc khi chỉ còn **một** lượt
   train phải chạy (không còn việc để song song hoá ở mức điều kiện).

**Nguồn đọc thêm**

- [Mixed Precision Training](https://arxiv.org/abs/1710.03740) — loss scaling và trọng số chủ, mục 3.
- [PyTorch: CUDA Automatic Mixed Precision](https://pytorch.org/docs/stable/notes/amp_examples.html) — GradScaler.
- [NVIDIA: Train With Mixed Precision](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/index.html)
  — fp16 so với bf16 trên từng thế hệ GPU.
- [An Empirical Model of Large-Batch Training](https://arxiv.org/abs/1812.06162) — gradient noise scale, batch tới hạn.
- [On the SDEs and Scaling Rules for Adaptive Gradient Algorithms](https://arxiv.org/abs/2205.10287) — nguồn của quy tắc căn cho Adam.
- [PyTorch Distributed: Experiences on Accelerating Data Parallel Training](https://arxiv.org/abs/2006.15704) — DDP.